
# Empirical penalty cap from short-term volatility (8% target variant)

A winning solver has an exclusivity window to settle an order. If the price moves against it by more than the penalty, reverting is cheaper than settling, so the cap decides how often reverts happen. This notebook sets the cap empirically:

> **cap(order) = k x sigma_1s(pair) x sqrt(T) x order size**

- `sigma_1s`: the pair's per-second volatility, measured with TSRV on tick data (noise-robust).
- `T`: the exclusivity window in seconds; volatility over the window grows with sqrt(T).
- `k`: the multiplier that hits the target revert rate. Read directly from the historical distribution of window price moves (a quantile), not from a parametric distribution.
- Order size: linear, since all quantities are in bps of order size.

**Data.** Binance spot tick data for ETHUSDT, SOLUSDT, DOGEUSDT, BNBUSDT, POLUSDT, AVAXUSDT and USDCUSDT, plus two correlated pairs: ETHBTC and WBETHETH (Binance's staked-ETH token, the closest listed proxy for wstETH/ETH; wstETH itself does not trade on Binance). Training: May 1-30 2026 (volatility and `k`). Testing: June 1 2026 to July 16 2026, held out; May 31 is also loaded to supply the first previous-day vol estimate. On the test days we count, per window, how often the end price falls below the expected band (start price minus the expected vol move)

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

T_EXCL = 26                        # measured mainnet exclusivity window (s)
T_GRID = [26, 12, 2]               # measured mainnet window, then one and two blocks shorter
Q_TARGET = 0.08                    # target revert rate

# ---- plotting: static plotly ----
pio.renderers.default = "png"
pio.renderers["png"].scale = 2
C_PAIR = {"ETHUSDT": "#2a78d6", "SOLUSDT": "#1baf7a", "DOGEUSDT": "#eb6834", "BNBUSDT": "#4a3aa7",
          "POLUSDT": "#8247e5", "AVAXUSDT": "#c22f2f", "USDCUSDT": "#c9a227",
          "ETHBTC": "#d02f8e", "WBETHETH": "#7a7f2a"}
PAIRS = list(C_PAIR)
GRAY, INK, GRID, SURF, AXIS = "#898781", "#0b0b0b", "#e1e0d9", "#fcfcfb", "#c3c2b7"

def style(fig, w=1050, h=430, title=None):
    fig.update_layout(width=w, height=h, template="none", paper_bgcolor=SURF, plot_bgcolor=SURF,
                      font=dict(size=13, color=INK), margin=dict(l=65, r=30, t=60, b=50),
                      legend=dict(bgcolor="rgba(0,0,0,0)"))
    if title: fig.update_layout(title=dict(text=title, font=dict(size=15)))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS)
    return fig
print("params set")

## 1. Data and volatility

- Trades are cleaned (duplicates dropped). 
- Volatility is estimated with TSRV method on 5-minute blocks

In [ ]:
import os, zipfile, urllib.request, pandas as pd

CACHE = os.path.expanduser("~/dev/penalty-research/data/binance_ticks")
os.makedirs(CACHE, exist_ok=True)
DAYS_TRAIN = [f"2026-05-{d:02d}" for d in range(1, 31)]    # 30 days
DAYS_TEST = ([f"2026-06-{d:02d}" for d in range(1, 31)]
             + [f"2026-07-{d:02d}" for d in range(1, 17)])   # June 1 - July 16, held out
WINDOW_VOL_S = 300

PERP_ONLY = set()          # pairs not listed on Binance spot would fall back to USD-M perp prints

def fetch(symbol, day):
    fn = f"{symbol}-aggTrades-{day}.zip"
    path = os.path.join(CACHE, fn)
    if not os.path.exists(path):
        seg = "futures/um/daily" if symbol in PERP_ONLY else "spot/daily"
        urllib.request.urlretrieve(
            f"https://data.binance.vision/data/{seg}/aggTrades/{symbol}/{fn}", path)
    return path

def load_ticks(symbol, days):
    frames = []
    for d in days:
        with zipfile.ZipFile(fetch(symbol, d)) as z:
            name = z.namelist()[0]
            hh = not z.open(name).read(64)[:1].isdigit()
            cols = ["aggId","price","qty","firstId","lastId","ts","isBuyerMaker","isBestMatch"]
            df = pd.read_csv(z.open(name), header=0 if hh else None, names=None if hh else cols)
            if hh:
                df.columns = [str(c).lower() for c in df.columns]
                tcol = [c for c in df.columns if "time" in c or c == "ts" or "transact" in c][0]
                df = df.rename(columns={tcol: "ts"})
            df = df[["price", "ts"]].astype({"ts": "int64", "price": float})
            unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"
            df["t"] = pd.to_datetime(df["ts"], unit=unit, utc=True)
            frames.append(df[["t", "price"]])
    return pd.concat(frames).sort_values("t").reset_index(drop=True)

def clean(df, k=25, mad_mult=3.0, floor_ticks=2.0, floor_bps=1.0):
    df = df[(df.price > 0) & df.price.notna()].drop_duplicates(subset=["t", "price"]).copy()
    tick = np.median(np.diff(np.unique(np.sort(df.price.values))))
    med = df.price.rolling(2*k + 1, center=True, min_periods=k).median()
    mad = (df.price - med).abs().rolling(2*k + 1, center=True, min_periods=k).median()
    floor_abs = max(floor_ticks*tick, df.price.median()*floor_bps*1e-4)
    keep = ((df.price - med).abs() <= mad_mult*mad + floor_abs).fillna(True)
    return df[keep].set_index("t")

def rv_all(lp):
    r = np.diff(lp); return float(r @ r)

def tsrv(lp, K):
    # two-scale realized variance, finite-sample adjusted
    n = len(lp) - 1
    if n < 2*K + 2: return np.nan
    rva = np.mean([np.sum(np.diff(lp[k::K])**2) for k in range(K)])
    nbar = (n - K + 1)/K
    return max((rva - (nbar/n)*rv_all(lp))/(1 - nbar/n), 0.0)

def prep(symbol, days):
    ticks = clean(load_ticks(symbol, days))
    px = ticks.price.resample("1s").last().ffill()
    return ticks, px

def block_vols(ticks, px):
    # per-5-min-block TSRV vol, per-second units, bps
    n_blocks = int((len(px) - 1)//WINDOW_VOL_S)
    edges = px.index[0] + pd.to_timedelta(np.arange(n_blocks + 1)*WINDOW_VOL_S, unit="s")
    pos = ticks.index.searchsorted(edges)
    lp_all = np.log(ticks.price.values)
    out = np.full(n_blocks, np.nan)
    for b in range(n_blocks):
        lp = lp_all[pos[b]:pos[b + 1]]
        if len(lp) < 200: continue
        Kw = max(2, min(int(round(len(lp)*10.0/WINDOW_VOL_S)), (len(lp) - 2)//2))
        out[b] = np.sqrt(tsrv(lp, Kw)/WINDOW_VOL_S)*1e4
    return out

# everything is computed per day so that vol and k can be re-estimated on a rolling basis
POOL_DAYS = DAYS_TRAIN + ["2026-05-31"] + DAYS_TEST     # May 1 - June 10, in order
DAY_PX, DAY_BLOCKS, SIGMA_1S, SIGMA_1S_PREV = {}, {}, {}, {}
for p in PAIRS:
    DAY_PX[p], DAY_BLOCKS[p] = {}, {}
    for d in POOL_DAYS:
        ticks, px = prep(p, [d])
        DAY_BLOCKS[p][d] = block_vols(ticks, px)
        DAY_PX[p][d] = px
        del ticks
    SIGMA_1S[p] = float(np.nanmedian(np.concatenate([DAY_BLOCKS[p][d] for d in DAYS_TRAIN])))
    SIGMA_1S_PREV[p] = {d: float(np.nanmedian(DAY_BLOCKS[p][prev]))
                        for prev, d in zip(POOL_DAYS[30:-1], DAYS_TEST)}
sigma_T = lambda pair, T: SIGMA_1S[pair]*np.sqrt(T)

print(f"{'pair':10s} {'sigma_1s May':>13s} {'prev-day vol range in June':>27s}   (bps/sqrt(s))")
for p in PAIRS:
    vs = list(SIGMA_1S_PREV[p].values())
    print(f"{p:10s} {SIGMA_1S[p]:13.3f} {min(vs):15.3f} - {max(vs):.3f}")

## 2. Window moves and the empirical k

Divide the training sample into non-overlapping windows of `T` seconds. Each window's move is the end price versus the start price, in bps: let's assume this is what a solver faces between winning and the settle deadline.

**Why moves are divided by vol.** `sigma_1s x sqrt(T)` is the typical (one standard deviation) price move for that pair over a `T`-second window: example: ~1.9 bps for ETHUSDT at 26s, ~4.1 bps for DOGEUSDT. Dividing each move by its pair's typical move converts it into "how unusual" is the current move relative to the standard deviation move: a raw -4 bps move is an ordinary window for DOGE (about 1x its typical move) but a bad one for ETH (over 2x). This separates the two components of the cap: the typical size (`sigma`) and the shape of the distribution (how often 1x, 2x, 3x typical moves happen), which is what `k` is.

`k` then comes straight from the data:

> `k = |8% quantile of window moves| / (sigma_1s x sqrt(T))`

i.e. the cap that keeps price-driven reverts to 8% of windows, expressed as a multiple of the pair's window volatility so it transfers across pairs and windows. No distributional assumption; the quantile is read off the sorted moves. The figure shows the move distributions with moves scaled by window vol, so pairs are comparable.

In [ ]:
def window_moves(px, T):
    v = px.values
    n = (len(v) - 1)//T
    p0 = v[: n*T: T]
    p1 = v[T: n*T + 1: T]
    return (p1/p0 - 1.0)*1e4     # bps

MV_DAY = {}
def moves_over(pair, days, T):
    # concatenated window moves over a list of days (windows never span a day boundary)
    return np.concatenate([MV_DAY.setdefault((pair, d, T), window_moves(DAY_PX[pair][d], T))
                           for d in days])

K_EMP = {p: {} for p in PAIRS}
print(f"{'pair':10s} {'T(s)':>5s} {'vol_T':>6s} {'q10 move':>9s} {'k':>6s}   (training)")
for p in PAIRS:
    for T in T_GRID:
        mv = moves_over(p, DAYS_TRAIN, T)
        q10 = float(np.quantile(mv, Q_TARGET))
        K_EMP[p][T] = -q10/sigma_T(p, T)
        print(f"{p:10s} {T:5d} {sigma_T(p, T):6.2f} {q10:9.2f} {K_EMP[p][T]:6.2f}")

fig = go.Figure()
for p in PAIRS:
    mv = moves_over(p, DAYS_TRAIN, T_EXCL)/sigma_T(p, T_EXCL)
    xs = np.sort(mv)[:: max(1, len(mv)//600)]
    fig.add_trace(go.Scatter(x=xs, y=np.linspace(0, 1, len(xs)), mode="lines",
                             line=dict(color=C_PAIR[p], width=2), name=p))
    fig.add_trace(go.Scatter(x=[-K_EMP[p][T_EXCL]], y=[Q_TARGET], mode="markers",
                             marker=dict(size=10, color=C_PAIR[p], symbol="diamond"),
                             showlegend=False))
from math import erf
xs_n = np.linspace(-4, 4, 200)
fig.add_trace(go.Scatter(x=xs_n, y=[0.5*(1 + erf(x/1.41421356)) for x in xs_n], mode="lines",
                         line=dict(color=INK, dash="dot", width=1.5), name="normal"))
for x in (-1, 1):
    fig.add_vline(x=x, line=dict(color=GRAY, dash="dot", width=1))
fig.add_hline(y=Q_TARGET, line=dict(color=INK, dash="dot", width=1))
fig.add_annotation(text="+-1 vol band", x=1.05, y=0.5, xanchor="left", showarrow=False,
                   font=dict(color=GRAY, size=11))
fig.add_annotation(text=f"diamonds: -k per pair at the {Q_TARGET:.0%} quantile", x=-3.7, y=0.16,
                   xanchor="left", showarrow=False, font=dict(size=11))
fig.update_xaxes(title=f"window move over {T_EXCL}s, in multiples of window vol", range=[-4, 4])
fig.update_yaxes(title="cumulative share of windows", tickformat=".0%")
style(fig, w=780, h=450, title=f"Distribution of {T_EXCL}s window moves (training, vol-scaled)")
fig.update_layout(legend=dict(x=0.02, y=0.97))
fig.show()

In [ ]:
bins = np.linspace(-4, 4, 33)
centers = (bins[:-1] + bins[1:])/2
fig = go.Figure()
for p in PAIRS:
    mv = moves_over(p, DAYS_TRAIN, T_EXCL)/sigma_T(p, T_EXCL)
    h, _ = np.histogram(mv, bins=bins, density=True)
    fig.add_trace(go.Scatter(x=centers, y=np.where(h > 0, h, np.nan), mode="lines",
                             line=dict(color=C_PAIR[p], width=2), name=p))
pdf = np.exp(-centers**2/2)/np.sqrt(2*np.pi)
fig.add_trace(go.Scatter(x=centers, y=pdf, mode="lines",
                         line=dict(color=INK, dash="dot", width=1.5), name="normal"))
fig.update_xaxes(title=f"window move over {T_EXCL}s, in multiples of window vol")
fig.update_yaxes(title="density")
style(fig, w=780, h=430, title=f"The same moves as a density ({T_EXCL}s, training, vol-scaled)")
fig.update_layout(legend=dict(x=0.02, y=0.97))
fig.show()

- In the cumulative chart, each curve is one pair's distribution of window moves in the training month: the price change from window start to end, divided by that pair's expected vol move (`sigma_1s x sqrt(T)`), so all pairs share one x-axis. A point reads: "y% of windows (y-axis) ended with a move below this many vols (x-axis)". The marker indicate where each curve crosses 8%: that x-position is the pair's `-k`, the cap in vol units breached in exactly 8% of training windows. The dotted curve is a normal distribution, which crosses 8% at -1.41

- The density chart shows the same moves as a distribution. It makes the shape difference visible.
- Why the quiet majors have the fattest scaled tails: DOGE delivers its variance in a steady churn, so a typical window already looks like an average one; ETH and BNB are calm most of the time and deliver a large share of their variance in bursts, so their bad windows sit far beyond a typical one. And since `sigma_1s` is the MEDIAN 5-minute block, `k` also prices how clustered a pair's volatility is relative to its everyday level, not just tail fatness.
    - Practical consequence: ETH needs its cap furthest out in vol multiples (k ~ 2.2 at the 8% quantile, well beyond the normal distribution's 1.41) yet its cap in bps stays among the smallest because its vol level is the lowest (at 26s: ETH 4.0 vs DOGE 6.0 bps).

- A caveat on the smaller pairs. POL and AVAX trade with a large minimum price increment relative to their volatility: one tick is several bps, versus <1 bp for the majors. Roughly half of their 26s windows show zero move and the rest land on whole-tick multiples (i.e. noise  or bounce); that is the tall spike at zero and the isolated bumps in their densities, and the staircase in their cumulative curves. USDC/USDT is the extreme case: its 8% quantile move is pinned to the tick grid at every window, so the cap floors at one tick and its k is just that tick divided by window vol.

## 3. Out of sample test (June 1 - July 16)

Windows are evaluated at the measured 26s mainnet exclusivity plus 12s and 2s (one and two blocks shorter solve deadlines); charts show the three windows as columns. At 2s the 1-second price grid and tick sizes dominate for all but the most liquid pairs, so treat that column as indicative.

1. **How often did prices drop more than one expected vol move?** For each window, check whether the end price finishes below the start minus one expected move (`sigma_1s x sqrt(T)`, calibrated in May). The bars compare the share of such windows in May itself with the share in the test period, against the same May band. Test bars far above May bars mean the market simply moved more than the calibration month. (Upside volatilit as well is found but not shown here, soJune brought bigger moves in both directions, not a drift.)
2. **Did the cap hold?** A revert happens when the end price falls below the start by more than the cap. Three versions of the cap, from most stale to most adaptive, all walk-forward (each day's cap is built only from data available before that day, so no look-forward bias):
   - **May k x May vol**: parameters frozen at the end of May.
   - **May k x latest 24h vol**: the multiplier stays fixed; `sigma_1s` is re-estimated each test day from the previous day's ticks, so the June 4 cap uses June 3 volatility.
   - **Rolling 30-day k x latest 24h vol**: `k` is also re-read every day, as the 8% quantile of vol-scaled window moves over the trailing 30 days, paired with the latest 24h vol.
   - **Rolling 30-day k x latest 1h vol**: same `k`, but the vol level is re-estimated window by window as the median of the previous hour's twelve 5-minute TSRV blocks: the fastest level refresh the block grid allows.
   - **Rolling 1-day k x latest 1h vol**: `k` re-read every day from the previous day's moves only (scaled by that day's vol), paired with the same 1h vol: a shorter `k` window, closer in cadence to the vol input.
   - **Constant May-calibrated cap**: one flat number per pair per window length, read off that pair's OWN May moves at the target percentile, then applied unchanged through the test period. By construction this coincides with `k[May] vol[May]` (that is exactly how the frozen cap is derived); it is shown under its own label so the fixed-bps option is directly comparable. This is the one-cap-for-all-pairs option; note a PER-PAIR May-calibrated constant is by construction identical to `k[May] vol[May]`, so that column doubles as the per-pair constant. Note this variant undershoots the target by roughly half: `k` was calibrated against the slow (30-day median) vol level, so it already prices how much bad hours exceed a typical hour. Pairing that `k` with a vol input that itself tracks the bad hours counts the clustering twice and over-covers. `k` and the vol input are a matched pair: whatever vol estimator runs in production must also be the one `k` is calibrated against.
   - **Direct percentile, 24h / 7d / 30d lookbacks**: the simpler alternative. Slice the recent past into `T`-second windows, rank the moves, and read the cap straight off the target percentile. No `k`, no volatility estimate, no `sqrt(T)`; one number per day per pair. Four lookbacks are tested: the previous hour (direct 1h, rolled forward window by window), the previous day (direct 24h), the trailing 7 days (direct 7d) and the trailing 30 days (direct 30d, the same window the rolling `k` uses, so the difference to that method is purely whether the vol level comes from the last 24h or is baked into the 30-day percentile). At 26s the 1h lookback has only ~138 windows, so its 8% percentile rests on about 11 observations. That noise barely hurts coverage, though: when the calibration window and the window being capped share the same distribution, the share of next moves below the k-th worst of n past moves is k/(n+1) regardless of the distribution, and an hour apart the two are nearly identical even while the day-scale regime moves. The cost of the 1h variant is not extra reverts but a cap level that jumps from hour to hour. On the same sample this is algebraically identical to `k x vol` (that is how `k` is defined), so the comparison tests how much data the percentile needs to be stable (~3,300 windows per day at 26s) against how quickly it adapts when the vol level moves.

In [ ]:
TEST, K_ROLL = {p: {} for p in PAIRS}, {p: {} for p in PAIRS}
CAP_TBL, REV_TBL = {}, {}          # per (pair, T): test-period cap and revert rate per method
# one constant per window length: calibrated on pooled May moves of the six liquid non-correlated
# pairs so that the pool breaches it at exactly the target rate; applied unchanged to every pair
# per-pair constant: each pair's own May target quantile, fixed through the test period
CONST_MAY = {(p, T): -float(np.quantile(moves_over(p, DAYS_TRAIN, T), Q_TARGET))
             for p in PAIRS for T in T_GRID}
DAY_RATES = {}
print("May-calibrated single constant (bps):", {T: round(c, 2) for T, c in CONST_MAY.items()})
DIR_CAP, DIR7_CAP, DIR30_CAP, ROLL_CAP = ({p: {} for p in PAIRS}, {p: {} for p in PAIRS},
                                          {p: {} for p in PAIRS}, {p: {} for p in PAIRS})
for p in PAIRS:
    for T in T_GRID:
        sT_tr = sigma_T(p, T)
        below_tr = float(np.mean(moves_over(p, DAYS_TRAIN, T) < -sT_tr))
        below_te, rev_may, rev_daily, rev_roll, rev_dir, rev_dir7, rev_dir30, ks, dcaps, d7caps, d30caps, rcaps = [], [], [], [], [], [], [], [], [], [], [], []
        for d in DAYS_TEST:
            i = POOL_DAYS.index(d)
            trail = POOL_DAYS[i - 30: i]                       # latest 30 days before d
            mv30 = moves_over(p, trail, T)
            sig30 = float(np.nanmedian(np.concatenate([DAY_BLOCKS[p][dd] for dd in trail])))
            k_r = -float(np.quantile(mv30, Q_TARGET))/(sig30*np.sqrt(T))
            ks.append(k_r)
            # direct method: rank past T-window moves, cap = the target percentile (24h and 7d lookbacks)
            cap_dir = -float(np.quantile(moves_over(p, [POOL_DAYS[i - 1]], T), Q_TARGET))
            cap_dir7 = -float(np.quantile(moves_over(p, POOL_DAYS[i - 7: i], T), Q_TARGET))
            cap_dir30 = -float(np.quantile(mv30, Q_TARGET))
            dcaps.append(cap_dir)
            d7caps.append(cap_dir7)
            d30caps.append(cap_dir30)
            mv = moves_over(p, [d], T)
            sT_prev = SIGMA_1S_PREV[p][d]*np.sqrt(T)
            below_te.append(mv < -sT_tr)
            rev_may.append(mv < -K_EMP[p][T]*sT_tr)
            rev_daily.append(mv < -K_EMP[p][T]*sT_prev)
            rev_roll.append(mv < -k_r*sT_prev)
            rev_dir.append(mv < -cap_dir)
            rev_dir7.append(mv < -cap_dir7)
            rev_dir30.append(mv < -cap_dir30)
            rcaps.append(k_r*sT_prev)
        K_ROLL[p][T] = (min(ks), max(ks))
        DIR_CAP[p][T] = float(np.mean(dcaps))
        DIR7_CAP[p][T] = float(np.mean(d7caps))
        DIR30_CAP[p][T] = float(np.mean(d30caps))
        ROLL_CAP[p][T] = float(np.mean(rcaps))
        below = float(np.mean(np.concatenate(below_te)))
        r_may = float(np.mean(np.concatenate(rev_may)))
        r_daily = float(np.mean(np.concatenate(rev_daily)))
        r_roll = float(np.mean(np.concatenate(rev_roll)))
        # direct (1h): rolling percentile of the previous hour's moves, refreshed window by window
        W = 3600//T
        full = moves_over(p, ["2026-05-31"] + DAYS_TEST, T)
        m0 = len(moves_over(p, ["2026-05-31"], T))
        caps1h = -pd.Series(full).rolling(W).quantile(Q_TARGET).to_numpy()[m0 - 1: -1]
        r_1h = float(np.mean(full[m0:] < -caps1h))
        # rolling k x latest 1h vol: same daily 30-day k, vol = median of the previous 12
        # five-minute TSRV blocks, refreshed window by window
        pool1h = ["2026-05-31"] + DAYS_TEST
        blk_all = [np.where(np.isnan(DAY_BLOCKS[p][dd]),
                            np.nanmedian(DAY_BLOCKS[p][dd]), DAY_BLOCKS[p][dd]) for dd in pool1h]
        blocks = np.concatenate(blk_all)
        off = np.concatenate([[0], np.cumsum([len(b) for b in blk_all])])
        med12 = np.median(np.lib.stride_tricks.sliding_window_view(blocks, 12), axis=1)
        rev_k1h, rev_k1d1h, caps_k1h, caps_k1d = [], [], [], []
        for j, d in enumerate(DAYS_TEST, start=1):
            mv_d = moves_over(p, [d], T)
            local_b = np.minimum((np.arange(len(mv_d))*T)//300, len(blk_all[j]) - 1)
            sig = med12[off[j] + local_b - 12]
            rev_k1h.append(mv_d < -ks[j - 1]*sig*np.sqrt(T))
            caps_k1h.append(ks[j - 1]*sig*np.sqrt(T))
            # k from the previous day only (quantile of its moves / its daily vol), applied with 1h vol
            mv_prev = moves_over(p, [POOL_DAYS[POOL_DAYS.index(d) - 1]], T)
            k_1d = -float(np.quantile(mv_prev, Q_TARGET))/(SIGMA_1S_PREV[p][d]*np.sqrt(T))
            rev_k1d1h.append(mv_d < -k_1d*sig*np.sqrt(T))
            caps_k1d.append(k_1d*sig*np.sqrt(T))
        r_k1h = float(np.mean(np.concatenate(rev_k1h)))
        r_k1d1h = float(np.mean(np.concatenate(rev_k1d1h)))
        mv_jun = full[m0:]
        r_cm = float(np.mean(mv_jun < -CONST_MAY[(p, T)]))
        r_dir = float(np.mean(np.concatenate(rev_dir)))
        r_dir7 = float(np.mean(np.concatenate(rev_dir7)))
        r_dir30 = float(np.mean(np.concatenate(rev_dir30)))
        n = sum(len(a) for a in below_te)
        TEST[p][T] = (below_tr, below, r_may, r_daily, r_roll, r_dir, r_dir7, r_dir30, r_1h, r_k1h, r_k1d1h, r_cm, n)
        CAP_TBL[(p, T)] = {
            "k[May] vol[May]": K_EMP[p][T]*sT_tr,
            "k[May] vol[24h]": float(np.mean([K_EMP[p][T]*SIGMA_1S_PREV[p][d]*np.sqrt(T) for d in DAYS_TEST])),
            "k[roll_30d] vol[24h]": float(np.mean(rcaps)),
            "k[roll_30d] vol[1h]": float(np.mean(np.concatenate(caps_k1h))),
            "k[roll_1d] vol[1h]": float(np.mean(np.concatenate(caps_k1d))),
            "direct[1h]": float(np.mean(caps1h)),
            "direct[24h]": float(np.mean(dcaps)),
            "direct[7d]": float(np.mean(d7caps)),
            "direct[30d]": float(np.mean(d30caps)),
            "const[May]": CONST_MAY[(p, T)]}
        REV_TBL[(p, T)] = {
            "k[May] vol[May]": r_may, "k[May] vol[24h]": r_daily,
            "k[roll_30d] vol[24h]": r_roll, "k[roll_30d] vol[1h]": r_k1h,
            "k[roll_1d] vol[1h]": r_k1d1h, "direct[1h]": r_1h, "direct[24h]": r_dir,
            "direct[7d]": r_dir7, "direct[30d]": r_dir30, "const[May]": r_cm}
        if T == T_EXCL:
            cuts = np.cumsum([len(moves_over(p, [d], T)) for d in DAYS_TEST])[:-1]
            DAY_RATES[p] = {
                "k[roll_1d] vol[1h]": np.array([float(a.mean()) for a in rev_k1d1h]),
                "direct[1h]": np.array([a.mean() for a in np.split(full[m0:] < -caps1h, cuts)]),
                "direct[24h]": np.array([float(a.mean()) for a in rev_dir]),
                "const[May]": np.array([a.mean() for a in np.split(mv_jun < -CONST_MAY[(p, T)], cuts)])}
from IPython.display import display
METHODS = ["k[May] vol[May]", "k[May] vol[24h]", "k[roll_30d] vol[24h]", "k[roll_30d] vol[1h]",
           "k[roll_1d] vol[1h]", "direct[1h]", "direct[24h]", "direct[7d]", "direct[30d]",
           "const[May]"]
rows_, idx_ = [], []
for p in PAIRS:
    for T in T_GRID:
        idx_.append((p, T))
        rows_.append({"below -1 May": TEST[p][T][0], "below -1 test": TEST[p][T][1],
                      **{m: REV_TBL[(p, T)][m] for m in METHODS}})
test_df = pd.DataFrame(rows_, index=pd.MultiIndex.from_tuples(idx_, names=["pair", "T"]))
print(f"Test-period (Jun 1 - Jul 16) revert rate per method (target {Q_TARGET:.0%}); below -1 = share of windows")
print("ending below start minus one expected move (May band). Method definitions in the bullets above.")
display(test_df.style.format("{:.1%}"))
print()
print(f"rolling k range across the test days, {T_EXCL}s window (May k for reference):")
for p in PAIRS:
    print(f"  {p:10s} {K_ROLL[p][T_EXCL][0]:.2f} - {K_ROLL[p][T_EXCL][1]:.2f}   (May: {K_EMP[p][T_EXCL]:.2f})")

fig = make_subplots(rows=1, cols=len(T_GRID), horizontal_spacing=0.06,
                    subplot_titles=[f"T = {t}s" for t in T_GRID])
for c0, t in enumerate(T_GRID):
    fig.add_trace(go.Bar(x=PAIRS, y=[TEST[p][t][0] for p in PAIRS], name="May (calibration month)",
                         marker_color="#b5b2ab", showlegend=(c0 == 0)), row=1, col=c0 + 1)
    fig.add_trace(go.Bar(x=PAIRS, y=[TEST[p][t][1] for p in PAIRS], name="Jun-Jul (held out)",
                         marker_color="#eb6834", showlegend=(c0 == 0)), row=1, col=c0 + 1)
    fig.update_yaxes(tickformat=".0%", row=1, col=c0 + 1)
    fig.update_xaxes(tickangle=60, row=1, col=c0 + 1)
fig.update_yaxes(title="share of windows", row=1, col=1)
style(fig, w=1500, h=480, title="Windows ending below -1 expected move")
fig.update_layout(barmode="group", margin=dict(t=120),
                  legend=dict(orientation="h", yanchor="bottom", y=1.03, x=0.0))
fig.show()

SERIES = [(2, "May k x May vol (frozen)", "#898781"),
          (3, "May k x latest 24h vol", "#2a78d6"),
          (4, "rolling 30-day k x latest 24h vol", "#1baf7a"),
          (9, "rolling 30-day k x latest 1h vol", "#0c6b4d"),
          (10, "rolling 1-day k x latest 1h vol", "#67c3a5"),
          (8, "direct (1h): percentile of last hour's moves", "#f3d391"),
          (5, "direct (24h): percentile of last day's moves", "#e0a63a"),
          (6, "direct (7d): percentile of last week's moves", "#8c5a16"),
          (7, "direct (30d): percentile of last 30 days' moves", "#59391c"),
          (11, "constant (each pair's own May quantile)", "#b5b2ab")]
fig = make_subplots(rows=1, cols=len(T_GRID), horizontal_spacing=0.05,
                    subplot_titles=[f"T = {t}s" for t in T_GRID])
for c0, t in enumerate(T_GRID):
    for k_idx, nm, col in SERIES:
        fig.add_trace(go.Bar(x=PAIRS, y=[TEST[p][t][k_idx] for p in PAIRS], name=nm,
                             marker_color=col, showlegend=(c0 == 0)), row=1, col=c0 + 1)
    fig.add_hline(y=Q_TARGET, line=dict(color=INK, dash="dot", width=1), row=1, col=c0 + 1)
    fig.update_yaxes(tickformat=".0%", row=1, col=c0 + 1)
    fig.update_xaxes(tickangle=60, row=1, col=c0 + 1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                         line=dict(color=INK, dash="dot", width=1), name=f"{Q_TARGET:.0%} target"))
fig.update_yaxes(title="share of windows", row=1, col=1)
style(fig, w=1700, h=560, title="Did the cap hold? Revert rates over the test period")
fig.update_layout(barmode="group", bargap=0.15, margin=dict(t=170),
                  legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0.0))
fig.show()

### The cap through July, window by window

The two leading contenders drawn against every realized 26s move for ETHUSDT in July. The shaded area is the downside cap in force at each moment; red markers are moves that breached it, i.e. would have been reverts under that cap.

In [ ]:
P_PLOT = "ETHUSDT"
days_p = [d for d in DAYS_TEST if d.startswith("2026-07")]
prev0 = POOL_DAYS[POOL_DAYS.index(days_p[0]) - 1]
mv_p = np.concatenate([moves_over(P_PLOT, [d], T_EXCL) for d in days_p])
n_day = len(moves_over(P_PLOT, [days_p[0]], T_EXCL))
tms = np.concatenate([(pd.date_range(d, periods=n_day, freq=f"{T_EXCL}s")
                       + pd.Timedelta(seconds=T_EXCL)).values for d in days_p])

# direct[1h]: rolling percentile of the previous 150 window moves
W = 3600//T_EXCL
full_p = moves_over(P_PLOT, [prev0] + days_p, T_EXCL)
m0 = len(moves_over(P_PLOT, [prev0], T_EXCL))
sw_p = np.lib.stride_tricks.sliding_window_view(full_p, W)
cap_dir_p = -np.quantile(sw_p[m0 - W: len(full_p) - W], Q_TARGET, axis=1)

# k[roll_1d] vol[1h]: k from the previous day, vol from the previous hour's TSRV blocks
pool_p = [prev0] + days_p
blk_p = [np.where(np.isnan(DAY_BLOCKS[P_PLOT][dd]), np.nanmedian(DAY_BLOCKS[P_PLOT][dd]),
                  DAY_BLOCKS[P_PLOT][dd]) for dd in pool_p]
blocks_p = np.concatenate(blk_p)
off_p = np.concatenate([[0], np.cumsum([len(b) for b in blk_p])])
med12_p = np.median(np.lib.stride_tricks.sliding_window_view(blocks_p, 12), axis=1)
cap_k_p = []
for j, d in enumerate(days_p, start=1):
    mv_d = moves_over(P_PLOT, [d], T_EXCL)
    local_b = np.minimum((np.arange(len(mv_d))*T_EXCL)//300, len(blk_p[j]) - 1)
    sig = med12_p[off_p[j] + local_b - 12]
    prev = pool_p[j - 1]
    k_1d = (-float(np.quantile(moves_over(P_PLOT, [prev], T_EXCL), Q_TARGET))
            / (float(np.nanmedian(DAY_BLOCKS[P_PLOT][prev]))*np.sqrt(T_EXCL)))
    cap_k_p.append(k_1d*sig*np.sqrt(T_EXCL))
cap_k_p = np.concatenate(cap_k_p)

def cap_plot(cap, name):
    br = mv_p < -cap
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=tms, y=-cap, mode="lines", line=dict(color="#e0a63a", width=0.6),
                             fill="tozeroy", fillcolor="rgba(224,166,58,0.30)", name="downside cap"))
    fig.add_trace(go.Scatter(x=tms, y=mv_p, mode="lines",
                             line=dict(color="#2a78d6", width=0.5), name=f"{T_EXCL}s window move"))
    fig.add_trace(go.Scatter(x=tms[br], y=mv_p[br], mode="markers",
                             marker=dict(color="#c22f2f", size=4.5), name="move beyond the cap"))
    fig.update_yaxes(title="bps")
    style(fig, w=1150, h=430,
          title=f"{P_PLOT}, July 1-16: {name} cap vs {T_EXCL}s moves; "
                f"breaches {int(br.sum()):,} of {len(mv_p):,} windows ({br.mean():.1%})")
    fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
    fig.show()

cap_plot(cap_k_p, "k[roll_1d] vol[1h]")
cap_plot(cap_dir_p, "direct[1h]")
cap_plot(np.full(len(mv_p), CONST_MAY[(P_PLOT, T_EXCL)]), f"constant {CONST_MAY[(P_PLOT, T_EXCL)]:.1f} bps (own May quantile)")

### Revert-rate variation over time

How much the realized revert rate breathes under each cap rule: daily revert rates for ETHUSDT at the 26s window, smoothed with a 7-day rolling mean, for the four leading rules. The spread around the target line is the size of the temporal variation under each method; the printed ranges give the unsmoothed daily extremes.

In [ ]:
sel = ["k[roll_1d] vol[1h]", "direct[1h]", "direct[24h]", "const[May]"]
cols = {"k[roll_1d] vol[1h]": "#67c3a5", "direct[1h]": "#f3d391",
        "direct[24h]": "#e0a63a", "const[May]": "#898781"}
dts = pd.to_datetime(DAYS_TEST)
fig = go.Figure()
for m in sel:
    r7 = pd.Series(DAY_RATES["ETHUSDT"][m], index=dts).rolling(7, min_periods=3).mean()
    fig.add_trace(go.Scatter(x=dts, y=r7, mode="lines", name=m, line=dict(color=cols[m], width=2)))
fig.add_hline(y=Q_TARGET, line=dict(color=INK, dash="dash", width=1.2))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                         line=dict(color=INK, dash="dash", width=1.2), name=f"{Q_TARGET:.0%} target"))
fig.update_yaxes(title="daily revert rate, 7-day rolling mean", tickformat=".0%")
style(fig, w=1000, h=430, title=f"ETHUSDT, {T_EXCL}s: revert rate over time under each cap rule")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
fig.show()
print("full-period mean and unsmoothed daily range:")
for m in sel:
    arr = DAY_RATES["ETHUSDT"][m]
    print(f"  {m:22s} mean {arr.mean():6.2%}   daily range {arr.min():.2%} - {arr.max():.2%}")

## 4. The cap table

One column group per cap-setting method. Under each method: `cap` = the test-period average of its walk-forward caps in bps of order size (for the frozen and constant variants this is just the fixed value), and `rev_rate` = its realized revert rate over the test days (Jun 1 - Jul 16). All caps multiply by notional to incorporate order size.

`const[May]` here is the PER-PAIR fixed-bps option: for each pair and window length, read the target percentile off that pair's own May moves and freeze the absolute value as the cap. The values are printed above the table. By construction this equals the `k[May] vol[May]` column (that is exactly how the frozen cap is built), so the two columns coincide; the constant label makes the fixed-bps option directly comparable across the table.

In [ ]:
from IPython.display import display

print(f"const[May] caps, per pair per window length (|{Q_TARGET:.0%} quantile| of the pair's OWN May moves):")
for p in PAIRS:
    print(f"   {p:10s} " + " | ".join(f"T = {T}s: {CONST_MAY[(p, T)]:6.2f} bps" for T in T_GRID))
print()
print(f"cap = test-period average of the walk-forward caps (bps); rev_rate = test-period revert rate; "
      f"target {Q_TARGET:.0%}")
METHODS = ["k[May] vol[May]", "k[May] vol[24h]", "k[roll_30d] vol[24h]", "k[roll_30d] vol[1h]",
           "k[roll_1d] vol[1h]", "direct[1h]", "direct[24h]", "direct[7d]", "direct[30d]",
           "const[May]"]
rows, idx = [], []
for p in PAIRS:
    for T in T_GRID:
        idx.append((p, T))
        r = {}
        for m in METHODS:
            r[(m, "cap")] = CAP_TBL[(p, T)][m]
            r[(m, "rev_rate")] = REV_TBL[(p, T)][m]
        rows.append(r)
cap_df = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(idx, names=["pair", "T"]))
cap_df.columns = pd.MultiIndex.from_tuples(cap_df.columns, names=["method", ""])
fmt = {c: ("{:.2f}" if c[1] == "cap" else "{:.1%}") for c in cap_df.columns}
display(cap_df.style.format(fmt))
print()
print("current fixed USD caps, converted to bps of order size (USD cap / order notional):")
print(f"{'chain':10s} {'USD cap':>8s} {'$1k order':>10s} {'$5k order':>10s} "
      f"{'$10k order':>11s} {'$100k order':>12s} {'$1M order':>10s}")
for usd, name in [(21.15, "ethereum"), (24.72, "bnb"), (2.62, "polygon")]:
    print(f"{name:10s} {usd:8.2f} {usd/1_000*1e4:6.0f} bps {usd/5_000*1e4:6.1f} bps "
          f"{usd/10_000*1e4:7.1f} bps {usd/100_000*1e4:8.2f} bps {usd/1_000_000*1e4:6.3f} bps")

## 5. Monte Carlo: bootstrapped months

To test how stable each method is across volatility regimes, we simulate months by resampling ACTUAL days: each simulated month is 30 days drawn at random, with replacement, from the 76 real days between May 2 and July 16, keeping every day's full intraday move sequence and volatility profile intact. Randomizing the day order is what stresses the methods: a calm day can follow a wild one, and vice versa, far more often than in the single real path.

- On each simulated month the leading walk-forward methods run against the path: `k[roll_1d] vol[1h]` and `direct[24h]` both recalibrate from whichever day the path sampled as yesterday (the k method via its multiplier, the direct method via the raw percentile), `direct[1h]` reads the cap off the trailing hour of moves, and the per-pair May-calibrated constant rides along as the fixed-bps benchmark.
- One approximation: the first hour of each simulated day is seeded by that day's real chronological neighbor rather than the sampled one; at most 1/24 of windows are affected.
- Each path yields one number per method: that month's realized revert rate. The histograms show the distribution across simulated months per asset and exclusivity window (columns); the dashed line is the target.

Three path designs are run, because how months are simulated decides which methods look good:

- **A. Daily resampling**: whole days drawn independently. By construction there is NO day-to-day correlation, which punishes the daily-calibrated methods with artificial regime jumps and flatters the hourly ones.
- **B. Hourly resampling**: single hours drawn independently. Now intra-day correlation is broken too, so every method calibrates on a slice unrelated to the one it is applied to. If design A's tilt toward hourly methods were an artifact, it should disappear here.
- **C. Volatility-persistent months**: real hours reordered by a fitted volatility process: log hourly vol is split into an hour-of-day seasonal profile plus deviations, the deviations get an AR(p) model with the lag order chosen by AIC (up to 24 hours; GARCH-family analogue), and simulated hours are mapped to real hours with the same clock hour and the nearest deviation. Months keep the true intraday shape, the fitted autocorrelation, and real empirical moves; evaluation is identical to design B, so B vs C isolates the effect of hour ordering (independent vs persistent).

In [ ]:
MC_PAIRS = list(PAIRS)
N_PATHS, N_DAYS = 3000, 30
rng = np.random.default_rng(7)
mc_days = POOL_DAYS[1:]                    # every day with a chronological predecessor
MC, DAYPRE, DAY_LSIG = {}, {}, {}
for T_MC in T_GRID:
    W_MC = 3600//T_MC
    for p in MC_PAIRS:
        nd = len(mc_days)
        k_day = np.empty(nd)               # k read from each day (used when sampled as yesterday)
        cap_day = np.empty(nd)             # direct[24h] cap read from each day (bps)
        rate_dir = np.empty(nd)            # direct[1h] revert rate of each day
        rate_cm = np.empty(nd)             # constant May-pooled cap revert rate of each day
        z_sorted, raw_sorted, n_win = [], [], np.empty(nd, dtype=int)
        for i, d in enumerate(mc_days):
            prev = POOL_DAYS[POOL_DAYS.index(d) - 1]
            mv_d = moves_over(p, [d], T_MC)
            blk_d = DAY_BLOCKS[p][d]
            blk_pv = DAY_BLOCKS[p][prev]
            blk_df = np.where(np.isnan(blk_d), np.nanmedian(blk_d), blk_d)
            blk_pf = np.where(np.isnan(blk_pv), np.nanmedian(blk_pv), blk_pv)
            blocks2 = np.concatenate([blk_pf, blk_df])
            m2 = np.median(np.lib.stride_tricks.sliding_window_view(blocks2, 12), axis=1)
            local_b = np.minimum((np.arange(len(mv_d))*T_MC)//300, len(blk_df) - 1)
            sig = m2[len(blk_pf) + local_b - 12]
            z_sorted.append(np.sort(mv_d/(sig*np.sqrt(T_MC))))
            raw_sorted.append(np.sort(mv_d))
            n_win[i] = len(mv_d)
            q_i = float(np.quantile(mv_d, Q_TARGET))
            cap_day[i] = -q_i
            k_day[i] = -q_i/(float(np.nanmedian(blk_d))*np.sqrt(T_MC))
            full2 = np.concatenate([moves_over(p, [prev], T_MC), mv_d])
            m02 = len(full2) - len(mv_d)
            capd = -pd.Series(full2).rolling(W_MC).quantile(Q_TARGET).to_numpy()[m02 - 1: -1]
            rate_dir[i] = float(np.mean(mv_d < -capd))
            rate_cm[i] = float(np.mean(mv_d < -CONST_MAY[(p, T_MC)]))
        RATE_K, RATE_D24 = np.empty((nd, nd)), np.empty((nd, nd))
        for j in range(nd):
            RATE_K[:, j] = np.searchsorted(z_sorted[j], -k_day)/n_win[j]
            RATE_D24[:, j] = np.searchsorted(raw_sorted[j], -cap_day)/n_win[j]
        idx = rng.integers(0, nd, size=(N_PATHS, N_DAYS + 1))
        MC[(p, T_MC)] = (RATE_K[idx[:, :-1], idx[:, 1:]].mean(axis=1),
                         rate_dir[idx[:, 1:]].mean(axis=1),
                         rate_cm[idx[:, 1:]].mean(axis=1),
                         RATE_D24[idx[:, :-1], idx[:, 1:]].mean(axis=1))
        DAYPRE[(p, T_MC)] = (RATE_K, RATE_D24, rate_dir, rate_cm)
        if T_MC == T_GRID[0]:
            dsig = np.array([float(np.nanmedian(DAY_BLOCKS[p][d])) for d in mc_days])
            dsig = np.where(np.isfinite(dsig) & (dsig > 0), dsig, np.nanmedian(dsig))
            DAY_LSIG[p] = np.log(dsig)
    print(f"T = {T_MC}s (mean over {N_PATHS:,} simulated months):")
    for p in MC_PAIRS:
        mc_k, mc_dir, mc_cm, mc_d24 = MC[(p, T_MC)]
        print(f"  {p:10s} k[roll_1d]x1h {mc_k.mean():6.2%} | direct[1h] {mc_dir.mean():6.2%} | "
              f"direct[24h] {mc_d24.mean():6.2%} | const[May] {mc_cm.mean():6.2%}")

In [ ]:
def mc_matrix(MCD, subtitle):
    nrows = len(MC_PAIRS)
    fig = make_subplots(rows=nrows, cols=len(T_GRID), horizontal_spacing=0.06, vertical_spacing=0.028,
                        column_titles=[f"T = {t}s" for t in T_GRID], row_titles=MC_PAIRS)
    for r0, p in enumerate(MC_PAIRS):
        for c0, t in enumerate(T_GRID):
            mc_k, mc_dir, mc_cm, mc_d24 = MCD[(p, t)]
            hi = float(max(mc_k.max(), mc_dir.max(), mc_cm.max(), mc_d24.max(), Q_TARGET*2))*1.05
            step = hi/40
            show = (r0 == 0 and c0 == 0)
            for arr, nm, col, op in ((mc_k, "k[roll_1d] vol[1h]", "#67c3a5", 0.65),
                                     (mc_dir, "direct[1h]", "#f3d391", 0.65),
                                     (mc_d24, "direct[24h]", "#e0a63a", 0.65),
                                     (mc_cm, "const[May]", "#898781", 0.55)):
                fig.add_trace(go.Histogram(x=arr, xbins=dict(start=0.0, end=hi, size=step), name=nm,
                                           marker_color=col, opacity=op, showlegend=show),
                              row=r0 + 1, col=c0 + 1)
            fig.add_vline(x=Q_TARGET, line=dict(color=INK, dash="dash", width=1.2), row=r0 + 1, col=c0 + 1)
            fig.update_xaxes(tickformat=".0%", row=r0 + 1, col=c0 + 1)
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                             line=dict(color=INK, dash="dash", width=1.2), name=f"{Q_TARGET:.0%} target"))
    style(fig, w=1250, h=250*nrows + 150,
          title=f"Simulated monthly revert rates ({subtitle})")
    fig.update_layout(barmode="overlay", margin=dict(t=130),
                      legend=dict(orientation="h", yanchor="bottom", y=1.012, x=0.0))
    fig.show()

mc_matrix(MC, f"A: daily resampling, {N_PATHS:,} months per asset and window")

### B. Hourly resampling

A simulated month is 720 hours drawn independently, with replacement, from the ~1,800 real hours in the pool. Every method calibrates on sampled history: `direct[1h]` on the previous sampled hour, `direct[24h]` and the 1-day `k` on the previous 24 sampled hours, so no method keeps any correlation advantage. Hour moves are represented by 128-point quantile sketches for speed; 1,000 paths per asset and window.

In [ ]:
N_PATHS_H = 1000
STEPS = N_DAYS*24
KQ = np.linspace(0, 1, 128)
MCH, HOURPRE = {}, {}

def eval_hour_paths(SK, SIG_H, RATE_H1, c_pt, seq, T_MC):
    # seq: (paths, 24 warmup + steps) hour indices; methods calibrate on sampled history
    Pn, total = seq.shape
    steps = total - 24
    sqT = np.sqrt(T_MC)
    r1 = RATE_H1[seq[:, 23:-1], seq[:, 24:]].mean(axis=1)
    KIDX = int(round(Q_TARGET*(24*128 - 1)))
    acc_k, acc_d, acc_c = np.zeros(Pn), np.zeros(Pn), np.zeros(Pn)
    for t in range(24, total):
        calib = seq[:, t - 24: t]
        q24 = np.partition(SK[calib].reshape(Pn, -1), KIDX, axis=1)[:, KIDX]
        sig_c = np.median(SIG_H[calib], axis=1)
        k1 = -q24/(sig_c*sqT)
        evs = SK[seq[:, t]]
        acc_d += (evs < q24[:, None]).mean(axis=1)
        acc_k += (evs < -(k1*SIG_H[seq[:, t - 1]]*sqT)[:, None]).mean(axis=1)
        acc_c += (evs < -c_pt).mean(axis=1)
    return acc_k/steps, r1, acc_c/steps, acc_d/steps

for T_MC in T_GRID:
    for p in MC_PAIRS:
        sks, sigs, hcl = [], [], []
        for d in mc_days:
            mv_d = moves_over(p, [d], T_MC)
            hrs = (np.arange(len(mv_d))*T_MC)//3600
            blk = DAY_BLOCKS[p][d]
            blk_f = np.where(np.isnan(blk), np.nanmedian(blk), blk)
            for h in range(24):
                mh = mv_d[hrs == h]
                if len(mh) < 10:
                    continue
                sks.append(np.quantile(mh, KQ))
                sigs.append(float(np.median(blk_f[h*12: min((h + 1)*12, len(blk_f))])))
                hcl.append(h)
        SK, SIG_H = np.array(sks), np.array(sigs)
        H = len(SK)
        caps_h = -SK[:, int(round(Q_TARGET*127))]
        RATE_H1 = np.empty((H, H))
        for j in range(H):
            RATE_H1[:, j] = np.searchsorted(SK[j], -caps_h)/128.0
        c_pt = CONST_MAY[(p, T_MC)]
        HOURPRE[(p, T_MC)] = (SK, SIG_H, RATE_H1, np.array(hcl), c_pt)
        idx = rng.integers(0, H, size=(N_PATHS_H, STEPS + 24))
        MCH[(p, T_MC)] = eval_hour_paths(SK, SIG_H, RATE_H1, c_pt, idx, T_MC)
    print(f"T = {T_MC}s (hourly resampling, means over {N_PATHS_H:,} months):")
    for p in MC_PAIRS:
        mc_k, mc_dir, mc_cm, mc_d24 = MCH[(p, T_MC)]
        print(f"  {p:10s} k[roll_1d]x1h {mc_k.mean():6.2%} | direct[1h] {mc_dir.mean():6.2%} | "
              f"direct[24h] {mc_d24.mean():6.2%} | const[May] {mc_cm.mean():6.2%}")

mc_matrix(MCH, f"B: hourly resampling, {N_PATHS_H:,} months per asset and window")

### C. Volatility-persistent months (hourly AR model)

Months built from real hours ordered by a fitted volatility process. Per pair: log hourly volatility is split into an hour-of-day seasonal profile plus deviations, and the deviations get an AR(p) model with the lag order chosen by AIC, up to 24 hours of memory; the chosen order and the coefficient sum (persistence) are printed. Each simulated hour is mapped back to a real hour with the SAME clock hour and the nearest volatility deviation, so months keep the true intraday shape, the fitted autocorrelation, and real empirical moves. Evaluation is identical to design B; B vs C isolates the effect of hour ordering, independent vs persistent.

In [ ]:
MCAR = {}
print("hourly log-vol model per pair: hour-of-day seasonal + AR(p), p by AIC (max 24):")
for p in MC_PAIRS:
    SK0, SIG0, R0, HCLK0, c0_ = HOURPRE[(p, T_GRID[0])]
    ls0 = np.log(np.where(SIG0 > 0, SIG0, np.nanmedian(SIG0[SIG0 > 0]) if (SIG0 > 0).any() else 1.0))
    seas = np.array([ls0[HCLK0 == h].mean() if (HCLK0 == h).any() else ls0.mean() for h in range(24)])
    x = ls0 - seas[HCLK0]
    n = len(x)
    best = None
    for q in range(1, 25):
        X = np.column_stack([x[q - 1 - j: n - 1 - j] for j in range(q)])
        y = x[q:]
        coef = np.linalg.lstsq(X, y, rcond=None)[0]
        rss = float(((y - X @ coef)**2).sum())
        aic = len(y)*np.log(max(rss, 1e-12)/len(y)) + 2*q
        if best is None or aic < best[0]:
            best = (aic, q, coef, float(np.sqrt(rss/len(y))))
    _, p_ar, coef, sd = best
    if not np.isfinite(sd) or sd == 0:
        p_ar, coef, sd = 1, np.array([0.0]), float(max(np.std(x), 1e-6))
    print(f"  {p:10s} p = {p_ar:2d}, coefficient sum = {coef.sum():5.2f}")
    for T_MC in T_GRID:
        SK, SIG_H, RATE_H1, HCLK, c_pt = HOURPRE[(p, T_MC)]
        lsT = np.log(np.where(SIG_H > 0, SIG_H, np.nanmedian(SIG_H[SIG_H > 0]) if (SIG_H > 0).any() else 1.0))
        devT = lsT - seas[HCLK]
        buckets, bvals = [], []
        for h in range(24):
            ih = np.where(HCLK == h)[0]
            if len(ih) == 0:
                ih = np.arange(len(devT))
            o = np.argsort(devT[ih])
            buckets.append(ih[o])
            bvals.append(devT[ih][o])
        total = STEPS + 24
        pos0 = rng.integers(p_ar, n, N_PATHS_H)
        hist = np.stack([x[pos0 - 1 - j] for j in range(p_ar)], axis=1)
        cstart = rng.integers(0, 24, N_PATHS_H)
        seq = np.empty((N_PATHS_H, total), dtype=int)
        for t in range(total):
            dev = (hist*coef[None, :]).sum(axis=1) + rng.normal(0.0, sd, N_PATHS_H)
            clock = (cstart + t) % 24
            for h in range(24):
                m = clock == h
                if not m.any():
                    continue
                pos = np.searchsorted(bvals[h], dev[m])
                pos = np.clip(pos + rng.integers(-2, 3, int(m.sum())), 0, len(bvals[h]) - 1)
                seq[m, t] = buckets[h][pos]
            hist = np.concatenate([dev[:, None], hist[:, :-1]], axis=1) if p_ar > 1 else dev[:, None]
        MCAR[(p, T_MC)] = eval_hour_paths(SK, SIG_H, RATE_H1, c_pt, seq, T_MC)
for T_MC in T_GRID:
    print(f"T = {T_MC}s (AR hourly vol-ordered, means over {N_PATHS_H:,} months):")
    for p in MC_PAIRS:
        mc_k, mc_dir, mc_cm, mc_d24 = MCAR[(p, T_MC)]
        print(f"  {p:10s} k[roll_1d]x1h {mc_k.mean():6.2%} | direct[1h] {mc_dir.mean():6.2%} | "
              f"direct[24h] {mc_d24.mean():6.2%} | const[May] {mc_cm.mean():6.2%}")

mc_matrix(MCAR, f"C: AR(p) hourly volatility-ordered, {N_PATHS_H:,} months per asset and window")

# Appendix

## 6. Steps for setting the Penalty cap

1. **Measure volatility.** TSRV per-second vol per pair on recent tick data; refresh on a rolling window.
2. **Take the window from config.** `T` = the chain's exclusivity window in seconds; vol over the window = `sigma_1s x sqrt(T)`.
3. **Pick the revert-rate target.** ex: 8% (per the Jul 17 discussion of anchoring near the current realized rate; this notebook's sibling `_2pct` version uses the 2% price-driven target agreed for production)
4. **Read `k` from the data.** The 8% quantile of historical window moves divided by window vol. No distributional assumption. One thing to think about here is whether to calculate 1 average k for similar assets (when data is sparse).
5. **Compute the cap:** `cap(order) = k x sigma_1s x sqrt(T) x order size`.

## 7. Optimal bid and revert rate as a function of the cap

So far the cap was set from the move distribution alone. This section adds the bidding side: given a cap, what is the zero-profit bid, and what revert rate does it imply. Everything is read off the empirical CDF of the May window moves, no distributional assumption.

- A solver that ties at bid `b` earns the realized move when it settles and pays at most the cap when it reverts, so its profit per unit of order size is `max(move - b, -cap)`. Setting the expectation to zero gives the equilibrium bid: `b* = E[max(move, b* - cap)]`.
- On the CDF this is an equal-areas condition: the area above the curve to the right of `b*` (expected profit) equals the area under the curve between `b* - cap` and `b*` (expected loss; flat further left because losses are capped). The expected revert rate is simply the height of the CDF at `b* - cap`.
- `b*` sits slightly above fair value plus the option value at fair value: adding the option value moves the truncation point right, which adds a little more option value, and so on; `b*` is the limit of that iteration. The gap is small, so fair value plus option value is a tight lower bound.
- External reverts (failures unrelated to price) enter as an insurance term: with probability `p_ext` the solver pays the cap no matter what, which lowers the bid by `p_ext/(1-p_ext) x cap`. Because the option premium itself is small, this term decides the sign of over- vs underbidding, so `p_ext` should be measured from settlement data rather than assumed.
- A cap of 0 is excluded: with nothing at risk the fixed point runs away (the free-option corner).

In [ ]:
def eq_bid(mv_sorted, prefix, cap, p_ext=0.0, iters=80):
    # fixed point of b = E[max(move, b - cap)] - p_ext/(1-p_ext)*cap, via sorted moves + prefix sums
    n = len(mv_sorted)
    total = prefix[-1]
    b = total/n
    for _ in range(iters):
        a = b - cap
        i = int(np.searchsorted(mv_sorted, a))
        emax = (a*i + (total - (prefix[i - 1] if i > 0 else 0.0)))/n
        b = emax - p_ext/(1.0 - p_ext)*cap
    return b

CAPS = list(range(1, 11))
SORT24, PRE24, FAIR24 = {}, {}, {}
for p in PAIRS:
    mv = np.sort(moves_over(p, DAYS_TRAIN, T_EXCL))
    SORT24[p], PRE24[p], FAIR24[p] = mv, np.cumsum(mv), float(np.mean(mv))

def revert_at(p, b, cap):
    return float(np.searchsorted(SORT24[p], b - cap))/len(SORT24[p])

def grid(fn, label):
    print(label)
    print(f"{'pair':10s}" + "".join(f"{c:>8d}" for c in CAPS) + "   (cap, bps)")
    for p in PAIRS:
        print(f"{p:10s}" + "".join(fn(p, c) for c in CAPS))
    print()

grid(lambda p, c: f"{revert_at(p, eq_bid(SORT24[p], PRE24[p], c), c):8.1%}",
     f"revert rate at the equilibrium bid (T = {T_EXCL}s, May training, p_ext = 0)")
grid(lambda p, c: f"{eq_bid(SORT24[p], PRE24[p], c) - FAIR24[p]:8.2f}",
     "bid premium b* - fair value, bps (p_ext = 0)")
grid(lambda p, c: f"{eq_bid(SORT24[p], PRE24[p], c, p_ext=0.03) - FAIR24[p]:8.2f}",
     "bid premium b* - fair value, bps (p_ext = 3%; insurance term is -0.031 x cap)")

d = max(abs(revert_at(p, eq_bid(SORT24[p], PRE24[p], c), c) - revert_at(p, FAIR24[p], c))
        for p in PAIRS for c in CAPS if c >= 3)
print(f"for caps of 3 bps and above, reading the revert rate at the FAIR bid instead changes")
print(f"the first table by at most {d:.2%}. At 1-2 bps caps on the tick-bound pairs the fixed")
print("point runs away (premiums of 1.7-5.3 bps): the model saying a cap below one tick is not viable.")
print("total revert rate including external reverts = p_ext + (1 - p_ext) x table value.")

Takeaways from the tables:

- The bid premium stays below ~0.5 bps across the whole 1-10 bps cap range and shrinks as the cap grows, so the choice of cap should be driven by the revert-rate table; the bidding side is second order.
- With `p_ext = 3%` the premium turns negative at moderate caps: the insurance term outweighs the option value, i.e. solvers would shade bids down rather than up. Whether that happens in practice depends on the real `p_ext`, which we should estimate from settlement data.

### The 3-second window

Mainnet is heading toward one auction per block, roughly 3 seconds of exposure. At that horizon the 1-second price grid and the tick size start to dominate for the smaller pairs, so the table below shows, per pair: the tick size, the share of 3s windows with zero price move, the cap read directly off the 3s distribution, and the cap scaled down from the 26s estimate with sqrt(time).

In [ ]:
T3 = 3
print(f"{'pair':10s} {'tick bps':>9s} {'zero @3s':>9s} {'direct cap @3s':>15s} "
      f"{'sqrt-scaled from ' + str(T_EXCL) + 's':>21s} {'ratio':>6s}")
for p in PAIRS:
    mv3 = moves_over(p, DAYS_TRAIN, T3)
    px = DAY_PX[p][DAYS_TRAIN[6]].values
    u = np.unique(px)
    tick = float(np.median(np.diff(u))/np.median(u))*1e4
    zero = float(np.mean(mv3 == 0.0))
    cap3 = -float(np.quantile(mv3, Q_TARGET))
    cap3_s = K_EMP[p][T_EXCL]*SIGMA_1S[p]*np.sqrt(T3)
    ratio = cap3/cap3_s if cap3_s > 0 else float("nan")
    print(f"{p:10s} {tick:9.2f} {zero:9.1%} {cap3:15.2f} {cap3_s:21.2f} {ratio:6.2f}")
print()
print(f"direct cap @3s = |{Q_TARGET:.0%} quantile of 3s moves| (May training).")
print(f"sqrt-scaled = k({T_EXCL}s) x sigma_1s x sqrt(3): the formula cap at the 3s horizon.")

Reading the 3s table:

- For POL and AVAX one tick is about 10 bps and the vast majority of 3s windows show zero move, so the whole 1-10 bps cap range sits inside one tick: the direct percentile is not readable there, and the revert-vs-cap curve is flat until it cliffs at one tick. This is also why caps for these pairs do not appear to scale with sqrt(time): the tick floor, not volatility dynamics. For tick-bound pairs the 3s cap should come from the formula, with an explicit floor of 1-2 ticks (a one-tick move is a real 10 bps loss, so the floor is economics, not a fudge).
- For POL and AVAX the 3s percentile is dominated by the ~10 bps tick, so the direct read is not usable there and the formula read with a tick floor is the reliable one at this horizon. For the liquid pairs the direct 3s read sits near the sqrt-scaled value, slightly below it where stale seconds deflate short-window moves.

Not covered here and next in line: estimating `p_ext` from settlement data, counterfactual per-solver penalties at an assumed cap over an accounting period, and re-running the calibration on additional months to check the stability of `k` and the caps.

## 8. Realized revert rates on Ethereum, per asset per day

Everything above prices reverts off CEX data; this section measures what actually happened on chain. Source: the penalties dataset extract (one row per auction x winning-solution order, from the analytics DB + Dune), filtered the same way as the penalties analysis notebook: fill-or-kill, in-market, penalty-eligible attempts, with a revert = the winner did not settle. The asset label is the non-stablecoin side of the pair (the sell side if neither is a stablecoin; unmapped token addresses are shown truncated; ETH and WETH are kept separate). The result is written to `ethereum_daily_reverts_per_asset.csv` next to this notebook, one row per (day, asset) plus a daily `_ALL` total.

Current extract coverage: 2026-01-01 to 2026-06-21. Refreshing beyond that needs the analytics DB credentials (the fetch script currently fails on the rotated password).

In [ ]:
import os

PATHS = [os.path.expanduser("~/dev/penalty-research/data/ethereum_2026-01-01_2026-06-22.csv"),
         "../data/ethereum_2026-01-01_2026-06-22.csv"]
SRC = next(p for p in PATHS if os.path.exists(p))
OUT = "ethereum_daily_reverts_per_asset.csv"      # written next to the notebook

TOKENS = {
    "0xeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee": "ETH",
    "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2": "WETH",
    "0x2260fac5e5542a773aa44fbcfedf7c193bc2c599": "WBTC",
    "0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf": "cbBTC",
    "0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0": "wstETH",
    "0xae7ab96520de3a18e5e111b5eaab095312d7fe84": "stETH",
    "0x514910771af9ca656af840dff83e8264ecf986ca": "LINK",
    "0x1f9840a85d5af5bf1d1762f925bdaddc4201f984": "UNI",
    "0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9": "AAVE",
    "0x6982508145454ce325ddbe47a25d4ec3d2311933": "PEPE",
    "0xfaba6f8e4a5e8ab82f62fe7c39859fa577269be3": "ONDO",
    "0xd533a949740bb3306d119cc777fa900ba034cd52": "CRV",
    "0x5afe3855358e112b5647b952709e6165e1c1eeee": "SAFE",
    "0xdef1ca1fb7fbcdc777520aa7f396b4e015f497ab": "COW",
    "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48": "USDC",
    "0xdac17f958d2ee523a2206206994597c13d831ec7": "USDT",
    "0x6b175474e89094c44da98b954eedeac495271d0f": "DAI",
    "0xdc035d45d973e3ec169d2276ddab16f1e407384f": "USDS",
    "0x4c9edd5852cd905f086c759e8383e09bff1e68b3": "USDe",
    "0x853d955acef822db058eb8505911ed77f175b99e": "FRAX",
    "0x6c3ea9036406852006290770bedfcaba0e23a0e8": "PYUSD",
}
STABLES = {"USDC", "USDT", "DAI", "USDS", "USDe", "FRAX", "PYUSD"}

cols = ["sell_token", "buy_token", "settled", "partially_fillable", "is_out_of_market",
        "is_excluded_from_penalties", "auction_timestamp"]
df = pd.read_csv(SRC, usecols=cols, low_memory=False)
for c in ["settled", "partially_fillable", "is_out_of_market", "is_excluded_from_penalties"]:
    df[c] = df[c].astype(str).str.lower().map({"true": True, "false": False})
df["date"] = pd.to_datetime(df["auction_timestamp"], errors="coerce", utc=True, format="mixed").dt.date
df["reverted"] = ~df["settled"].fillna(False)

# in-market fill-or-kill, penalty-eligible attempts only (same filters as the penalties analysis notebook)
df = df[(df["partially_fillable"] == False) & (df["is_out_of_market"] == False)
        & (df["is_excluded_from_penalties"] == False)].copy()

def sym(addr):
    a = str(addr).lower()
    return TOKENS.get(a, a[:6] + ".." + a[-4:] if a.startswith("0x") else a)

sell = df["sell_token"].map(sym)
buy = df["buy_token"].map(sym)
# asset = the non-stablecoin side of the pair; sell side if neither is a stablecoin
df["asset"] = np.where(sell.isin(STABLES) & buy.isin(STABLES), "STABLE/STABLE",
              np.where(sell.isin(STABLES), buy, sell))

g = (df.groupby(["date", "asset"])
       .agg(n_attempts=("reverted", "size"), n_reverts=("reverted", "sum"))
       .reset_index())
g["revert_rate"] = g["n_reverts"] / g["n_attempts"]
tot = (df.groupby("date")
         .agg(n_attempts=("reverted", "size"), n_reverts=("reverted", "sum"))
         .reset_index())
tot["asset"] = "_ALL"
tot["revert_rate"] = tot["n_reverts"] / tot["n_attempts"]
out = pd.concat([g, tot[g.columns]], ignore_index=True).sort_values(
    ["date", "n_attempts"], ascending=[True, False])
out.to_csv(OUT, index=False)
print(f"source: {SRC}")
print(f"wrote {OUT}: {len(out):,} rows, {out['date'].min()} to {out['date'].max()}, "
      f"{out[out.asset != '_ALL']['asset'].nunique():,} assets")
print("top assets by attempts:")
top = out[out.asset != "_ALL"].groupby("asset")[["n_attempts", "n_reverts"]].sum()
top["revert_rate"] = top["n_reverts"]/top["n_attempts"]
print(top.sort_values("n_attempts", ascending=False).head(12).to_string())